# 06: Ground Truth Dataset Upload

This notebook loads the curated ground truth dataset from
`data/ground_truth_dataset.jsonl` and uploads it to Langfuse so the
offline evaluation suite (Notebook 07) has concrete rubrics to judge against.

## What's in the ground truth dataset

Each example provides everything the LLM judges need — not just the expected
text answer, but the full evaluation rubric:

| Field | Evaluator that reads it | What it provides |
|---|---|---|
| `expected_output` | DeepSearchQA (F1 / Outcome) | Ground truth answer text |
| `plan_rubric.must_cover` | Plan Quality | Specific concepts the plan must address |
| `tool_pattern.reference_tool_call_count` | Tool Selection / Efficiency | Expected call count from an expert run |
| `tool_pattern.must_use` | Tool Selection / Appropriateness | Tools the agent must invoke |
| `source_rubric.expected_tiers` | Source Validation | Authority tier expected for this category |
| `max_replan_threshold` | Replanning Rate | Allowed replannings before flagging |
| `requires_knowledge_base` | KB Usage | Whether `vertex_search` must be called |
| `kb_concepts` | KB Usage | Concepts that should appear in the KB answer |

## Examples in this dataset

9 examples spanning 8 categories (Finance & Economics, Statistics & Data,
Science & Technology, Politics & Government, Health & Medicine,
History & Culture, Environment & Climate).

## Prerequisites

Complete Notebooks 01–05. Credentials in `.env`:
- `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY`

In [ ]:
import json
import os
from pathlib import Path

from aieng.agent_evals.async_client_manager import AsyncClientManager
from dotenv import load_dotenv
from rich.console import Console
from rich.panel import Panel
from rich.table import Table


if Path("").absolute().name == "eval-agents":
    print(f"Working directory: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"Working directory set to: {Path('').absolute()}")

load_dotenv(verbose=True)
console = Console(width=100)

GROUND_TRUTH_PATH = Path("implementations/knowledge_qa/data/ground_truth_dataset.jsonl")
DATASET_NAME = "KnowledgeQA-GroundTruth"

## 1. Load and Inspect the Ground Truth Dataset

Each line in the JSONL is a self-contained evaluation example.
Let's load them and preview the schema.

In [ ]:
examples = []
with open(GROUND_TRUTH_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#"):
            examples.append(json.loads(line))

console.print(f"Loaded [bold]{len(examples)}[/bold] ground truth examples from {GROUND_TRUTH_PATH}")

t = Table(title="Ground Truth Dataset Overview")
t.add_column("ID", style="yellow", justify="right", width=4)
t.add_column("Category", style="cyan", width=26)
t.add_column("Answer Type", style="dim", width=14)
t.add_column("KB?", style="magenta", width=4)
t.add_column("must_cover concepts", style="white")
t.add_column("Ref calls", style="dim", justify="right", width=9)

for ex in examples:
    meta = ex["metadata"]
    pr = meta.get("plan_rubric", {})
    tp = meta.get("tool_pattern", {})
    kb = "[green]yes[/green]" if meta.get("requires_knowledge_base") else "no"
    must_cover = pr.get("must_cover", [])
    t.add_row(
        str(meta["example_id"]),
        meta["category"],
        meta["answer_type"],
        kb,
        ", ".join(must_cover[:2]) + (" ..." if len(must_cover) > 2 else ""),
        str(tp.get("reference_tool_call_count", "auto")),
    )
console.print(t)

## 2. Inspect a Single Example

Let's look at the full annotation for each example to understand what the judges receive.

In [ ]:
# Example 3 — Statistics & Data (high complexity, Tier 2 sources required)
ex = examples[2]  # 0-indexed
console.print(Panel(
    f"[bold cyan]Question:[/bold cyan] {ex['input']}\n\n"
    f"[bold yellow]Expected answer:[/bold yellow] {ex['expected_output']}\n\n"
    f"[bold]plan_rubric:[/bold] {json.dumps(ex['metadata']['plan_rubric'], indent=2)}\n\n"
    f"[bold]tool_pattern:[/bold] {json.dumps(ex['metadata']['tool_pattern'], indent=2)}\n\n"
    f"[bold]source_rubric:[/bold] {json.dumps(ex['metadata']['source_rubric'], indent=2)}",
    title=f"Example {ex['metadata']['example_id']} — {ex['metadata']['category']}",
    border_style="blue",
))

In [ ]:
# Example 9 — Environment & Climate (IPCC AR6, Tier 1 sources required)
ex9 = next(ex for ex in examples if ex["metadata"]["example_id"] == 9)
console.print(Panel(
    f"[bold cyan]Question:[/bold cyan] {ex9['input']}\n\n"
    f"[bold yellow]Expected answer:[/bold yellow] {ex9['expected_output']}\n\n"
    f"[bold]plan_rubric:[/bold] {json.dumps(ex9['metadata']['plan_rubric'], indent=2)}\n\n"
    f"[bold]tool_pattern:[/bold] {json.dumps(ex9['metadata']['tool_pattern'], indent=2)}\n\n"
    f"[bold]source_rubric:[/bold] {json.dumps(ex9['metadata']['source_rubric'], indent=2)}",
    title=f"Example 9 — {ex9['metadata']['category']}",
    border_style="blue",
))

## 3. Upload to Langfuse

Each example becomes a Langfuse dataset item:
- `input` → the question
- `expected_output` → ground truth answer text (used by the DeepSearchQA F1 grader)
- `metadata` → full rubric (used by Plan Quality, Tool Selection, Source Validation, KB evaluators)

If an item with the same `input` already exists in the dataset, it will be updated
with the latest metadata — safe to re-run.

In [ ]:
client_manager = AsyncClientManager.get_instance()
langfuse = client_manager.langfuse_client

# Create the dataset if it doesn't exist yet
langfuse.create_dataset(
    name=DATASET_NAME,
    description="9-example curated ground truth with full evaluation rubrics for Knowledge QA offline eval",
)

console.print(f"Uploading [bold]{len(examples)}[/bold] examples to Langfuse dataset [bold cyan]{DATASET_NAME!r}[/bold cyan]...")

for ex in examples:
    example_id = ex["metadata"]["example_id"]
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        id=f"KnowledgeQA-gt-{example_id}",  # stable ID — re-runs update rather than duplicate
        input=ex["input"],
        expected_output=ex["expected_output"],
        metadata=ex["metadata"],
    )

langfuse.flush()
console.print(f"[green]✓[/green] Upload complete — {len(examples)} items in dataset {DATASET_NAME!r}")

## 4. Verify: Read Back from Langfuse

Confirm the items landed in Langfuse with the metadata intact.

In [ ]:
dataset = langfuse.get_dataset(DATASET_NAME)
items = dataset.items

console.print(f"Dataset [bold cyan]{DATASET_NAME!r}[/bold cyan] contains [bold]{len(items)}[/bold] items")

t = Table(title="Langfuse Dataset Items (verification)")
t.add_column("ID", style="yellow", justify="right", width=4)
t.add_column("Category", style="cyan", width=26)
t.add_column("KB?", style="magenta", width=4)
t.add_column("must_cover count", style="dim", justify="right", width=16)
t.add_column("Ref calls", style="dim", justify="right", width=9)
t.add_column("Question (preview)", style="white")

for item in sorted(items, key=lambda x: x.metadata.get("example_id", 0)):
    meta = item.metadata or {}
    pr = meta.get("plan_rubric", {})
    tp = meta.get("tool_pattern", {})
    q = str(item.input)
    kb = "[green]yes[/green]" if meta.get("requires_knowledge_base") else "no"
    t.add_row(
        str(meta.get("example_id", "?")),
        meta.get("category", "?"),
        kb,
        str(len(pr.get("must_cover", []))),
        str(tp.get("reference_tool_call_count", "auto")),
        q[:55] + "..." if len(q) > 55 else q,
    )
console.print(t)

## 5. Annotation Schema Reference

When adding new examples to `data/ground_truth_dataset.jsonl`,
use this schema. Only include fields relevant to your question — all
rubric fields are optional and fall back to auto-derived defaults.

In [ ]:
SCHEMA = {
    "input": "<question text>",
    "expected_output": "<ground truth answer>",
    "metadata": {
        "example_id": "<unique integer>",
        "category": "<Finance & Economics | Statistics & Data | Science & Technology | ...",
        "answer_type": "<Single Answer | Set Answer>",
        "plan_rubric": {
            "must_cover": ["concept 1", "concept 2"],  # REQUIRED for Plan Quality
            # optional overrides (auto-derived from category/answer_type if omitted):
            # "min_steps": 2,
            # "max_steps": 4,
            # "complexity": "low | medium | high",
            # "must_have_synthesis": False,
        },
        "tool_pattern": {
            "reference_tool_call_count": 3,  # RECOMMENDED for Tool Selection
            # optional overrides:
            # "must_use": ["google_search", "web_fetch"],
            # "may_use": ["vertex_search"],
            # "data_type": "web | structured | mixed",
        },
        # source_rubric is FULLY auto-derived from category — omit unless overriding
        # max_replan_threshold is FULLY auto-derived from answer_type/category — omit unless overriding
        # KB-only fields (only needed when requires_knowledge_base is True):
        "requires_knowledge_base": False,
        "kb_concepts": [],  # concepts the KB answer must mention
    }
}

console.print(Panel(
    json.dumps(SCHEMA, indent=2),
    title="Annotation Schema",
    border_style="dim",
))
console.print(
    "[dim]Add new lines to data/ground_truth_dataset.jsonl following this schema,\n"
    "then re-run this notebook to upload them to Langfuse.[/dim]"
)

## Summary

In this notebook you:

1. **Loaded** 9 ground truth examples from `data/ground_truth_dataset.jsonl`,
   spanning 8 categories (Finance & Economics, Statistics & Data, Science & Technology,
   Politics & Government, Health & Medicine, History & Culture, Environment & Climate)
2. **Inspected** the full annotation schema — `must_cover`, `reference_tool_call_count`,
   `source_rubric`, and `requires_knowledge_base`
3. **Created** the `KnowledgeQA-GroundTruth` dataset in Langfuse (idempotent — safe to re-run)
4. **Uploaded** all examples with stable IDs so re-runs update existing items rather than duplicating
5. **Verified** that all items landed with the correct metadata
6. **Reviewed** the schema for adding new annotated examples

### What's next

- **Notebook 07** — run the full offline evaluation suite against this dataset.
  The LLM judges will now have `must_cover`, `reference_tool_call_count`, and
  `source_rubric` to anchor their scoring, producing much more specific feedback
  than auto-derived rubrics alone.
- To add more examples: append lines to `data/ground_truth_dataset.jsonl`
  and re-run this notebook — existing items are updated, not duplicated.